# Ejercicio 2: zona, codificación y disponibilidad

El mismo `viajes_taxi.csv`, sin los errores confirmados. La última celda imprime una comprobación por cada `TODO`; no la modifiques. Completa los `TODO` hasta que las cuatro digan `OK`, y después pasa a las preguntas del README.

Corre todas las celdas una vez antes de empezar (**Run All**).

In [1]:
import pandas as pd

viajes = pd.read_csv("viajes_taxi.csv")
viajes = viajes[(viajes["tarifa"] != 999.99) & (viajes["duracion_min"] >= 0)].copy()

ZONAS_MANHATTAN = ["Midtown", "Upper East Side", "Upper West Side", "Lower Manhattan", "Harlem"]

## Paso 1: la tarifa fija como columna

In [2]:
# DONE 1: crea la columna es_tarifa_fija (1 o 0): vale 1 cuando el viaje
# va de JFK Airport a una zona de Manhattan, o de una zona de Manhattan a
# JFK Airport. Imprime cuantos viajes la tienen en 1, y la tarifa minima
# y maxima de esos viajes.
desde_jfk = (viajes["zona_origen"] == "JFK Airport") & viajes["zona_destino"].isin(ZONAS_MANHATTAN)
hacia_jfk = viajes["zona_origen"].isin(ZONAS_MANHATTAN) & (viajes["zona_destino"] == "JFK Airport")
viajes["es_tarifa_fija"] = (desde_jfk | hacia_jfk).astype(int)
fijos = viajes[viajes["es_tarifa_fija"] == 1]
print("Viajes con tarifa fija:", len(fijos))
print("Tarifa minima:", fijos["tarifa"].min(), "| maxima:", fijos["tarifa"].max())

Viajes con tarifa fija: 300
Tarifa minima: 70.0 | maxima: 70.0


## Paso 2: zona de origen con one-hot

In [3]:
# DONE 2: codificado, el resultado de codificar zona_origen con one-hot
# (pd.get_dummies), con el prefijo "origen" y dtype=int. Imprime los
# nombres de las columnas nuevas.
codificado = pd.get_dummies(viajes["zona_origen"], prefix="origen", dtype=int)
print(list(codificado.columns))

['origen_Brooklyn', 'origen_Harlem', 'origen_JFK Airport', 'origen_Lower Manhattan', 'origen_Midtown', 'origen_Queens', 'origen_Upper East Side', 'origen_Upper West Side']


## Paso 3: zona de origen ordinal

In [4]:
# DONE 3: codifica zona_origen de forma ordinal: un numero entero por
# zona, en orden alfabetico, en una columna origen_ordinal.
zonas_en_orden = sorted(viajes["zona_origen"].unique())
viajes["origen_ordinal"] = viajes["zona_origen"].map({zona: i for i, zona in enumerate(zonas_en_orden)})
print(dict(enumerate(zonas_en_orden)))

{0: 'Brooklyn', 1: 'Harlem', 2: 'JFK Airport', 3: 'Lower Manhattan', 4: 'Midtown', 5: 'Queens', 6: 'Upper East Side', 7: 'Upper West Side'}


## Paso 4: qué existe al pedir el taxi

In [5]:
# DONE 4: disponible_al_pedir, un diccionario con cada columna original
# del dataset y True o False segun exista en el momento en que alguien
# pide el taxi (antes del viaje). Imprime la decision de cada una.
# distancia_km se estima con el destino que da el pasajero al pedir; la
# duracion, la tarifa cobrada, el metodo de pago y la propina solo se
# conocen al terminar el viaje.
disponible_al_pedir = {
    "id_viaje": True,
    "fecha_hora_inicio": True,
    "zona_origen": True,
    "zona_destino": True,
    "distancia_km": True,
    "duracion_min": False,
    "pasajeros": True,
    "metodo_pago": False,
    "tarifa": False,
    "propina": False,
}
for columna, disponible in disponible_al_pedir.items():
    print(f"{columna:18s} {'disponible' if disponible else 'NO disponible'} al pedir")

id_viaje           disponible al pedir
fecha_hora_inicio  disponible al pedir
zona_origen        disponible al pedir
zona_destino       disponible al pedir
distancia_km       disponible al pedir
duracion_min       NO disponible al pedir
pasajeros          disponible al pedir
metodo_pago        NO disponible al pedir
tarifa             NO disponible al pedir
propina            NO disponible al pedir


## Comprobación

In [6]:
def comprobar(numero, condicion):
    if condicion:
        print(f"TODO {numero}: OK")
    else:
        print(f"TODO {numero}: revisa")


comprobar(1, "es_tarifa_fija" in viajes.columns and viajes["es_tarifa_fija"].sum() == 300)
comprobar(2, codificado is not None
          and len([columna for columna in codificado.columns if columna.startswith("origen_")]) == 8)
comprobar(3, "origen_ordinal" in viajes.columns
          and viajes.loc[viajes["zona_origen"] == "Brooklyn", "origen_ordinal"].iloc[0] == 0
          and viajes["origen_ordinal"].max() == 7)
comprobar(4, disponible_al_pedir is not None
          and disponible_al_pedir.get("duracion_min") is False
          and disponible_al_pedir.get("propina") is False
          and disponible_al_pedir.get("distancia_km") is True)

TODO 1: OK
TODO 2: OK
TODO 3: OK
TODO 4: OK
